# sr-diffusion x4 prototype demo

This notebook runs the public `jwheo/sr-diffusion` prototype checkpoint from Hugging Face.

Use **Runtime -> Change runtime type -> GPU** before running. The checkpoint is a research prototype under a non-commercial license, not a polished production SR model.

In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## 1. Clone the repo and install lightweight dependencies

Colab already provides PyTorch, so this avoids reinstalling `torch`.

In [ ]:
import os
import subprocess
from pathlib import Path

os.chdir('/content')
repo = Path('sr-diffusion')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/BitIntx/sr-diffusion.git'], check=True)
os.chdir(repo)
subprocess.run(['git', 'pull', '--ff-only'], check=True)
print('repo:', Path.cwd())

In [ ]:
%pip -q install pyyaml pillow huggingface_hub
%pip -q install -e . --no-deps

## 2. Download public prototype checkpoints

This downloads the VAE, Stage 2 condition encoder, Stage 3 baseline, and Stage 4 condition-start checkpoint.

In [ ]:
!python scripts/download_hf_checkpoints.py

## 3. Upload an image

For real LR inference, upload a 128x128-ish LR image. Larger images are center-cropped/resized to the expected LR size by default.

For a controlled smoke test, upload an HR image and set `INPUT_MODE = 'hr'` in the next cell. The script will crop/degrade it first.

In [ ]:
from google.colab import files

uploaded = files.upload()
INPUT_IMAGE = next(iter(uploaded))
print('input:', INPUT_IMAGE)

## 4. Run x4 SR

In [ ]:
import subprocess
from pathlib import Path

INPUT_MODE = 'lr'  # 'lr' for a low-resolution input, 'hr' for controlled HR->LR->SR smoke test
OUTPUT_DIR = Path('outputs/colab_demo')
STEPS = 32
SEED = 123

input_flag = '--input-lr' if INPUT_MODE == 'lr' else '--input-hr'
cmd = [
    'python', 'infer_diffusion.py',
    input_flag, INPUT_IMAGE,
    '--output-dir', str(OUTPUT_DIR),
    '--steps', str(STEPS),
    '--seed', str(SEED),
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

## 5. View and download the result

In [ ]:
from IPython.display import display
from PIL import Image
from pathlib import Path

out = Path('outputs/colab_demo')
print('LR input')
display(Image.open(out / 'input_lr.png'))
gt = out / 'gt_hr.png'
if gt.exists():
    print('GT HR')
    display(Image.open(gt))
print('SR output')
display(Image.open(out / 'sr_00.png'))

In [ ]:
from google.colab import files
files.download('outputs/colab_demo/sr_00.png')

## Optional: compare Stage 3 baseline

The default demo uses `configs/hf/diffusion_stage4_condition.yaml`. To compare the earlier Stage 3 baseline, pass `--config configs/hf/diffusion_stage3_baseline.yaml` to `infer_diffusion.py`.